In [ ]:
# 1. Prepare the Workbench
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from google.colab import drive

# Mount the Drive to access our Master Dataset
print("Unlocking the vault...")
drive.mount('/content/drive')

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-9qaacv2y/unsloth_7f65068e0de54ebca986ac8aa076c6c9
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-9qaacv2y/unsloth_7f65068e0de54ebca986ac8aa076c6c9
  Resolved https://github.com/unslothai/unsloth.git to commit 2ca3f660c087c2702a70641d22f46d07fb09c63d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 39.0 MB/s eta 0:00:00
   ━━━━

In [ ]:

# 2. Summon the Base Model
max_seq_length = 2048
dtype = None # Auto-detects fp16 for T4
load_in_4bit = True # Absolute necessity for the free T4 GPU

print("Summoning the Qwen-3B architecture...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. Forge the LoRA Adapters
# This allows us to train a 3-billion parameter model on a free GPU by only updating 1-2% of the weights.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Summoning the Qwen-3B architecture...
==((====))==  Unsloth 2026.8.15: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.8.15 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
# 4. Ingest the Grand Ledger (Our 40-Book Dataset)
master_path = "/content/drive/MyDrive/VictorianGPT/dialogues/victorian_dataset_master.json"
dataset = load_dataset("json", data_files=master_path, split="train")

# We must format our input/output pairs into Qwen's expected conversational template
def formatting_prompts_func(examples):
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for input_text, output_text in zip(inputs, outputs):
        text = f"<|im_start|>user\n{input_text}<|im_end|>\n<|im_start|>assistant\n{output_text}<|im_end|>"
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Dataset successfully mapped! Preparing to train on {len(dataset)} Victorian interactions.")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/46313 [00:00<?, ? examples/s]

Dataset successfully mapped! Preparing to train on 46313 Victorian interactions.


In [ ]:
# 5. The Training Crucible (Architected for 40 Novels)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,  # Keep this small (2) so the T4 GPU doesn't melt
        gradient_accumulation_steps = 8,  # Accumulates to an effective batch size of 16
        warmup_steps = 50,
        num_train_epochs = 1,             # With 40 books, 1 full epoch is more than enough
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/content/drive/MyDrive/VictorianGPT/checkpoints",

        # CRITICAL SAFETY MEASURES FOR HUGE DATASETS:
        save_strategy = "steps",
        save_steps = 100,                 # Saves a backup every 100 steps
        save_total_limit = 2,             # Only keeps the last 2 backups to save Drive space
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/46313 [00:00<?, ? examples/s]

In [ ]:
# 6. Ignite the Furnace!
print("Igniting the fine-tuning furnace. This will take time. Mind the heat...")
trainer_stats = trainer.train()

# 7. Secure the Final Artifact
final_save_path = "/content/drive/MyDrive/VictorianGPT/Victorian_Qwen_Final_Adapter"
model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)

print(f"\nGlorious success! The mind of our Victorian scholar is permanently sealed in:\n{final_save_path}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Igniting the fine-tuning furnace. This will take time. Mind the heat...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 46,313 | Num Epochs = 1 | Total steps = 2,895
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,4.707746
20,3.924174
30,3.166275
40,2.963548
50,2.875712
60,2.866712
70,2.897533
80,2.808604
90,2.813879
100,2.875635


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/VictorianGPT/checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/VictorianGPT/checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/VictorianGPT/checkpoints/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/VictorianGPT/checkpoints/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/VictorianGPT/checkpoints/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/VictorianGPT/checkpoints/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/VictorianGPT/checkpoints/checkpoint-700/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder me


Glorious success! The mind of our Victorian scholar is permanently sealed in:
/content/drive/MyDrive/VictorianGPT/Victorian_Qwen_Final_Adapter
